# Tutorial 2 — Write a detection once for every cloud (25 minutes)

Run all cells once. This notebook uses the **completed reference mappings** so it works even
if you have not finished Lab 2. It does not edit participant SQL files or attach the UI database.
You will inspect OCSF fields, compare one normalized rule with three raw rules, and write a variation.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts at the repository root or inside tutorials/.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "workshop" / "db.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start JupyterLab from the workshop checkout: ./setup.sh --notebooks")
sys.path.insert(0, str(ROOT))
from workshop.tutorial import open_workshop, q, compare, evaluate

con = open_workshop(mappings="solutions")


## 1. Learn the normalized schema
`metadata.uid` is the event ID; `cloud.provider` is AWS/GCP/Azure; `actor.user.name`
is the source principal; `api.operation` is the Kubernetes verb. `resources[1]` is the first
resource (SQL LIST indexes start at 1). Pod exec becomes resource type `pods/exec`.
Nested `api.request.data` retains the Kubernetes pod spec as JSON.


In [ ]:
q(con, """
SELECT cloud.provider, api.operation, resources[1].type AS resource_type, count(*) AS events
FROM ocsf_api_activity GROUP BY ALL ORDER BY events DESC
""")


In [ ]:
q(con, "DESCRIBE ocsf_api_activity", 30)


## 2. A complete portable privileged-pod rule
We require a successful create (`status_id = 1`). Kubernetes has several container lists:
this baseline intentionally checks only `spec.containers[0]`, matching the existing raw exercise.
You will expand that coverage below. The remaining projection includes enrichment join keys.


In [ ]:
PRIVILEGED_SQL = """
SELECT metadata.uid AS event_id, cloud.provider AS provider, time,
       actor.user.name AS principal, resources[1].namespace AS namespace,
       json_extract_string(api.request.data, '$.spec.containers[0].image') AS image,
       json_extract_string(api.request.data, '$.metadata.ownerReferences[0].name') AS owner_name,
       json_extract_string(api.request.data, '$.metadata.ownerReferences[0].kind') AS owner_kind,
       cloud.account.uid AS account_uid,
       cloud.account.labels['cluster_name'] AS cluster_name,
       src_endpoint.ip AS src_ip, http_request.user_agent AS user_agent
FROM ocsf_api_activity
WHERE api.operation = 'create'
  AND resources[1].type = 'pods'
  AND status_id = 1
  AND json_extract_string(api.request.data, '$.spec.containers[0].securityContext.privileged') = 'true'
"""
q(con, PRIVILEGED_SQL)


In [ ]:
q(con, f"SELECT provider, count(*) AS hits FROM ({PRIVILEGED_SQL}) GROUP BY ALL ORDER BY 1")


## 3. Check equivalence to the existing raw rules
Exact ID equality checks portability on this fixture dataset. It does not demonstrate that every
hit is malicious, nor guarantee coverage of every real Kubernetes audit format.


In [ ]:
from solutions.lab1_raw.rules import PRIVILEGED_POD
RAW_REFERENCE = " UNION ALL ".join(f"SELECT event_id FROM ({sql})" for sql in PRIVILEGED_POD.values())
result = compare(con, PRIVILEGED_SQL, RAW_REFERENCE)
assert result["match"], "Inspect missing/extra event IDs before proceeding"


## 4. Your rule: pod exec across all three clouds
Start with the complete example. Then duplicate the practice cell and add filters for a namespace,
principal, or time range. Run the comparison to see how much the narrower rule drops.
No cloud-specific JSON paths are necessary for this rule.


In [ ]:
EXEC_SQL = """
SELECT metadata.uid AS event_id, cloud.provider AS provider, time,
       actor.user.name AS principal, resources[1].namespace AS namespace,
       resources[1].name AS pod, src_endpoint.ip AS src_ip
FROM ocsf_api_activity
WHERE api.operation = 'create' AND resources[1].type = 'pods/exec'
"""
q(con, EXEC_SQL)


In [ ]:
MY_EXEC_SQL = EXEC_SQL  # YOUR QUERY: replace this with your own SQL string.
q(con, MY_EXEC_SQL)


In [ ]:
compare(con, MY_EXEC_SQL, EXEC_SQL)


In [ ]:
evaluate(con, MY_EXEC_SQL)


## 5. Worked extension: inspect every regular container
`json_each` expands the containers array; `EXISTS` avoids duplicating the event if two containers
are privileged. Edit the JSON path to investigate `initContainers` too, or add a separate condition
for `hostPID`. A hostPID-only workload is a different rule and should have a different name.
Missing or null fields must not crash a detection.


In [ ]:
ALL_CONTAINERS_SQL = """
SELECT o.metadata.uid AS event_id, o.cloud.provider AS provider,
       o.actor.user.name AS principal, o.resources[1].namespace AS namespace
FROM ocsf_api_activity o
WHERE o.api.operation = 'create' AND o.resources[1].type = 'pods' AND o.status_id = 1
  AND EXISTS (
    SELECT 1 FROM json_each(json_extract(o.api.request.data, '$.spec.containers')) c
    WHERE json_extract_string(c.value, '$.securityContext.privileged') = 'true'
  )
"""
q(con, ALL_CONTAINERS_SQL)


In [ ]:
compare(con, ALL_CONTAINERS_SQL, PRIVILEGED_SQL)


This dataset may produce the same hits: that alone does not test the new multi-container case.
Here is a tiny regression fixture with a privileged **second** container.


In [ ]:
second_container_test = con.sql("""
WITH example AS (
  SELECT json '{"spec":{"containers":[{"securityContext":{"privileged":false}},
                                       {"securityContext":{"privileged":true}}]}}' AS payload
)
SELECT EXISTS (
  SELECT 1 FROM json_each(json_extract(payload, '$.spec.containers')) c
  WHERE json_extract_string(c.value, '$.securityContext.privileged') = 'true'
) AS detected FROM example
""").fetchone()[0]
assert second_container_test
print("Second-container regression fixture passed")


**Checkpoint:** keep an `event_id` column in your custom rule. Document what it detects,
required fields, likely benign causes, and information missing from a cloud's logs.
GKE does not record the exec command URI here, so normalization cannot recover command arguments.

Optional mapper challenge: edit `labs/lab2_ocsf/*.sql` and run the Lab 2 tests in `WORKSHOP.md`.
Then change the setup cell to `open_workshop(mappings="participant")` and rerun the notebook.
Continue with `03_enrichment_and_tuning.ipynb`.


In [ ]:
# Add your next query below: q(con, "SELECT ...")
